In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import wandb

# ==========================================
# 0. WANDB INIT
# ==========================================
wandb.init(
    project="quasar-flare-gru-pinn",
    name="gru_ou",
    config={
        "epochs": 100,
        "lr": 1e-3,
        "sigma_floor": 0.02,
        "lambda_phys_start": 0.005,
        "lambda_phys_end": 0.05,
        "lambda_var_start": 0.01,
        "lambda_var_end": 0.20,
        "hidden_dim": 64
    }
)

config = wandb.config

# ==========================================
# 1. MODEL
# ==========================================
class QuasarProbabilisticGRU(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.gru = nn.GRU(input_size=2, hidden_size=hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, 2)

    def forward(self, x, dt):
        rnn_input = torch.cat((x, dt), dim=-1)
        out, _ = self.gru(rnn_input)
        params = self.head(out)

        mu_pred = params[:, :, 0:1]
        sigma_pred = F.softplus(params[:, :, 1:2]) + config.sigma_floor
        return mu_pred, sigma_pred


# ==========================================
# 2. LOSS FUNCTION
# ==========================================
def refined_ou_loss(mu_p, sigma_p, y_true, dt,
                    taus, sigmas_ou, obj_means,
                    lambd_phys, lambd_var):

    y_target = y_true[:, 1:, :]
    dt_step = dt[:, 1:, :]
    y_prev = y_true[:, :-1, :]

    sigma_p = torch.clamp(sigma_p, min=0.02, max=5.0)

    # --- Probabilistic NLL ---
    nll = 0.5 * torch.log(sigma_p**2) + \
          0.5 * ((y_target - mu_p)**2 / sigma_p**2)
    data_loss = torch.mean(nll)

    # --- OU physics terms ---
    tau_exp = taus.unsqueeze(1)
    sig_ou_exp = sigmas_ou.unsqueeze(1)

    expected_mu = obj_means + (y_prev - obj_means) * torch.exp(-dt_step / tau_exp)

    expected_var = (sig_ou_exp**2 * tau_exp / 2) * \
                   (1 - torch.exp(-2 * dt_step / tau_exp))

    expected_var = torch.clamp(expected_var, min=0.01)

    drift_loss = torch.mean(
        (mu_p - expected_mu)**2 / (expected_var + 1e-8)
    )

    var_reg_loss = torch.mean(
        (sigma_p**2 - expected_var)**2
    )

    total = data_loss + lambd_phys*drift_loss + lambd_var*var_reg_loss
    return total, data_loss, drift_loss, var_reg_loss


# ==========================================
# 3. DATASET
# ==========================================
class QuasarDataset(Dataset):
    def __init__(self, folder_path, metadata_df):
        self.folder_path = folder_path
        self.metadata = metadata_df

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        file_path = os.path.join(self.folder_path, f"{int(row['dbID'])}.csv")

        df = pd.read_csv(file_path).sort_values("MJD")

        mjd = torch.tensor(df["MJD"].values).float()
        mag = torch.tensor(df["mag_r_sim"].values).float()

        # mean centering
        mag = mag - mag.mean()

        dt = torch.cat([torch.tensor([0.0]), mjd[1:] - mjd[:-1]])
        dt = torch.clamp(dt, min=1e-3)

        return {
            "mag": mag.unsqueeze(-1),
            "dt": dt.unsqueeze(-1),
            "tau": torch.tensor([row["tau"]]).float(),
            "sig_ou": torch.tensor([row["sigma"]]).float(),
            "mu_obj": torch.tensor([0.0]).float()
        }


# ==========================================
# 4. TRAINING SETUP
# ==========================================
SIM_DATA_PATH = r'data\simulations'
METADATA_PATH = r'data\metadata.csv'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

meta_df = pd.read_csv(METADATA_PATH)

loader = DataLoader(
    QuasarDataset(SIM_DATA_PATH, meta_df),
    batch_size=1,
    shuffle=True
)

model = QuasarProbabilisticGRU().to(device)
optimizer = optim.Adam(model.parameters(), lr=config.lr)

wandb.watch(model, log="gradients", log_freq=500)

# ==========================================
# 5. LOCAL HISTORY (CSV SAVE)
# ==========================================
history = {
    "epoch": [],
    "loss_total": [],
    "loss_data": [],
    "loss_phys": [],
    "loss_var": [],
    "lambda_phys": [],
    "lambda_var": []
}

# ==========================================
# 6. TRAIN LOOP
# ==========================================
epochs = config.epochs

for epoch in range(epochs):

    progress = epoch / (epochs - 1) if epochs > 1 else 1.0

    lp = config.lambda_phys_start + \
         progress * (config.lambda_phys_end - config.lambda_phys_start)

    lv = config.lambda_var_start + \
         progress * (config.lambda_var_end - config.lambda_var_start)

    model.train()

    run_total = run_data = run_phys = run_var = 0
    valid_batches = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in pbar:

        mag = batch["mag"].to(device)
        dt  = batch["dt"].to(device)

        if mag.size(1) < 2:
            continue

        tau = batch["tau"].to(device)
        sig_ou = batch["sig_ou"].to(device)
        mu_obj = batch["mu_obj"].to(device)

        optimizer.zero_grad()

        mu_p, sigma_p = model(mag[:, :-1, :], dt[:, :-1, :])

        loss, l_data, l_phys, l_var = refined_ou_loss(
            mu_p, sigma_p, mag, dt,
            tau, sig_ou, mu_obj,
            lp, lv
        )

        if not torch.isfinite(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        valid_batches += 1
        run_total += loss.item()
        run_data += l_data.item()
        run_phys += l_phys.item()
        run_var += l_var.item()

        if valid_batches % 100 == 0:
            pbar.set_postfix({
                "L": f"{run_total/valid_batches:.2f}",
                "lp": f"{lp:.2f}",
                "lv": f"{lv:.2f}"
            })

    # ===== Epoch averages =====
    if valid_batches > 0:
        avg_total = run_total / valid_batches
        avg_data  = run_data / valid_batches
        avg_phys  = run_phys / valid_batches
        avg_var   = run_var / valid_batches

        # ---- LOCAL CSV STORAGE ----
        history["epoch"].append(epoch + 1)
        history["loss_total"].append(avg_total)
        history["loss_data"].append(avg_data)
        history["loss_phys"].append(avg_phys)
        history["loss_var"].append(avg_var)
        history["lambda_phys"].append(lp)
        history["lambda_var"].append(lv)

        # ---- WANDB LOGGING ----
        wandb.log({
            "epoch": epoch + 1,
            "loss_total": avg_total,
            "loss_data": avg_data,
            "loss_phys": avg_phys,
            "loss_var": avg_var,
            "lambda_phys": lp,
            "lambda_var": lv,
            "lr": optimizer.param_groups[0]["lr"]
        })

        print(f"Epoch {epoch+1}: L={avg_total:.3f}")

    # Save model each epoch
    torch.save(model.state_dict(), "quasar_hybrid_gru_model_v5.pth")

# ==========================================
# 7. SAVE CSV HISTORY
# ==========================================
pd.DataFrame(history).to_csv("training_history.csv", index=False)
print("Saved training history to training_history.csv")

print("Training complete.")
wandb.finish()
